# Fusion Retrieval: BM25 + Dense Retrieval + Reciprocal Rank Fusion

[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/cyruslayo/castalia/blob/main/rag/fusion_retrieval.ipynb)

## The Single Highest-Impact RAG Improvement

Every retrieval system has a fundamental tension: **exact keyword matching** vs. **semantic understanding**. A user searching for *"COVID-19 transmission rates"* needs a system that can:

1. **Match the exact term** "COVID-19" (not just "coronavirus" or "pandemic")
2. **Understand the concept** of "transmission rates" (even if a document says "how fast the disease spreads")

No single retrieval method handles both well. **Sparse retrieval** (BM25) excels at exact matching but misses synonyms. **Dense retrieval** (embeddings + vector search) captures semantics but can miss critical keywords.

**Fusion retrieval** combines both, and **Reciprocal Rank Fusion (RRF)** is the elegant algorithm that merges their ranked results without needing score calibration.

### What This Notebook Covers

| Section | Topic |
|---------|-------|
| §1 | Sparse vs Dense Retrieval — two paradigms, two blind spots |
| §2 | BM25 from First Principles — the math behind term matching |
| §3 | Dense Retrieval Internals — bi-encoders and semantic space |
| §4 | Why Fusion? — concrete failure cases that motivate hybrid search |
| §5 | Reciprocal Rank Fusion — the RRF formula and why it works |
| §6 | Implementation from Scratch — BM25, FAISS, RRF pipeline |
| §7 | Head-to-Head Comparison — BM25 vs Dense vs Fusion |
| §8 | Weighted Fusion — tuning the sparse/dense balance |
| §9 | Complete RAG Pipeline — fusion retrieval + generation |
| §10 | Synthesis — production patterns and when to use fusion |

## 🎯 Learning Objectives

By the end of this notebook, you will:

- Understand **The Single Highest-Impact RAG Improvement**
- Understand **Sparse Retrieval (BM25)**
- Understand **Dense Retrieval (Embeddings)**
- Understand **The Evolution: Boolean → TF-IDF → BM25**
- Understand **The BM25 Formula**


<div style="text-align: center;">

<img src="../images/fusion_retrieval.svg" alt="Fusion Retrieval" style="width:100%; height:auto;">
</div>

In [ ]:
# Install all dependencies
!pip install -q transformers>=4.51.0 accelerate bitsandbytes torch sentence-transformers faiss-cpu numpy rank-bm25

In [ ]:
import os, torch, warnings
warnings.filterwarnings('ignore')

from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig
from sentence_transformers import SentenceTransformer

# ── Cache paths (Google Drive for Colab persistence) ──
from google.colab import drive
drive.mount("/content/drive")

CACHE_DIR = "/content/drive/MyDrive/models"
os.makedirs(CACHE_DIR, exist_ok=True)

# ── Load Qwen3-8B in 4-bit NF4 ──
LLM_ID = "Qwen/Qwen3-8B"
bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.bfloat16,
    bnb_4bit_use_double_quant=True,
)

tokenizer = AutoTokenizer.from_pretrained(LLM_ID, cache_dir=CACHE_DIR)
model = AutoModelForCausalLM.from_pretrained(
    LLM_ID,
    quantization_config=bnb_config,
    device_map="auto",
    cache_dir=CACHE_DIR,
)

# ── Load embedding model ──
EMBED_ID = "BAAI/bge-base-en-v1.5"
embed_model = SentenceTransformer(EMBED_ID, cache_folder=CACHE_DIR)

print(f"✓ LLM loaded: {LLM_ID} (4-bit NF4)")
print(f"✓ Embeddings loaded: {EMBED_ID} (dim={embed_model.get_sentence_embedding_dimension()})")
print(f"✓ Device: {model.device}")

In [ ]:
def generate(prompt: str, max_new_tokens: int = 512, temperature: float = 0.7) -> str:
    """Generate text from the loaded Qwen3 model."""
    messages = [{"role": "user", "content": prompt}]
    text = tokenizer.apply_chat_template(
        messages, tokenize=False, add_generation_prompt=True,
        enable_thinking=False   # skip reasoning tokens for RAG answers
    )
    inputs = tokenizer(text, return_tensors="pt").to(model.device)
    with torch.no_grad():
        output = model.generate(
            **inputs,
            max_new_tokens=max_new_tokens,
            temperature=temperature,
            top_p=0.9,
            do_sample=True,
        )
    return tokenizer.decode(
        output[0][inputs['input_ids'].shape[1]:], skip_special_tokens=True
    ).strip()

# Quick sanity check
print(generate("What is retrieval-augmented generation in one sentence?"))

---
# §1 — Sparse vs Dense Retrieval: Two Paradigms, Two Blind Spots

Every retrieval system must answer one question: **given a query, which documents are relevant?**

Two fundamentally different approaches have emerged:

## Sparse Retrieval (BM25)

**Representation:** Each document is a sparse vector in vocabulary space. Most dimensions are zero — only terms that appear in the document have nonzero weights.

**Matching:** A document is relevant if it contains the *exact query terms*. Relevance is scored by term frequency, document length, and how rare the term is across the corpus.

**Strengths:**
- Perfect at exact keyword matching: product IDs, medical codes, proper nouns
- Fast, interpretable, no training required
- Works well out of the box on any domain

**Blind spots:**
- Cannot match synonyms: "car" ≠ "automobile"
- No understanding of meaning: "bank" (river) vs "bank" (financial)
- Struggles with paraphrases: "how to fix a flat tire" vs "repairing a punctured wheel"

## Dense Retrieval (Embeddings)

**Representation:** Each document is a dense vector in a learned embedding space (e.g., 768 dimensions from a transformer). Every dimension is nonzero.

**Matching:** Relevance is measured by cosine similarity or inner product between query and document embeddings. Semantically similar texts cluster together.

**Strengths:**
- Understands meaning: "car" ≈ "automobile" ≈ "vehicle"
- Handles paraphrases and conceptual similarity
- Works across languages (with multilingual models)

**Blind spots:**
- Can miss exact terms: "COVID-19" might not surface if training data used "coronavirus"
- Rare terms and codes get diluted in the embedding
- Needs training data; domain mismatch degrades quality

In [ ]:
# Visual comparison of sparse vs dense retrieval characteristics
comparison = {
    "Criterion":        ["Exact keywords", "Synonyms", "Paraphrases", "Rare terms/codes",
                          "Semantic similarity", "No training needed", "Interpretable"],
    "BM25 (Sparse)":    ["★★★★★", "★☆☆☆☆", "★☆☆☆☆", "★★★★★",
                          "★☆☆☆☆", "★★★★★", "★★★★★"],
    "Dense (Embeddings)":["★★☆☆☆", "★★★★★", "★★★★★", "★★☆☆☆",
                           "★★★★★", "★☆☆☆☆", "★★☆☆☆"],
}

print(f"{'Criterion':<22} {'BM25 (Sparse)':<16} {'Dense (Embeddings)'}")
print("─" * 56)
for i, criterion in enumerate(comparison['Criterion']):
    print(f"{criterion:<22} {comparison['BM25 (Sparse)'][i]:<16} {comparison['Dense (Embeddings)'][i]}")

print("\n→ Neither method dominates. Fusion combines the best of both.")

---
# §2 — BM25 from First Principles

## The Evolution: Boolean → TF-IDF → BM25

**Boolean search** (1960s): A document either contains a term or doesn't. No ranking.

**TF-IDF** (1972): Introduced two key insights:
- **Term Frequency (TF):** A term appearing 5 times in a document is more indicative than appearing once.
- **Inverse Document Frequency (IDF):** A term appearing in 3 out of 10,000 documents is more discriminating than one appearing in 9,000.

**BM25** (1994, Robertson & Walker): The *probabilistic* refinement of TF-IDF, based on the Binary Independence Model. BM25 adds two critical corrections:
1. **Saturating TF:** Beyond a point, more occurrences add diminishing returns (a term appearing 100× isn't 100× more relevant than appearing once).
2. **Document length normalization:** Long documents naturally contain more term occurrences; BM25 normalizes for this.

## The BM25 Formula

$$\text{score}(q, d) = \sum_{t \in q} \text{IDF}(t) \cdot \frac{\text{tf}(t, d) \cdot (k_1 + 1)}{\text{tf}(t, d) + k_1 \cdot \left(1 - b + b \cdot \frac{|d|}{\text{avgdl}}\right)}$$

### Term-by-term breakdown:

| Symbol | Meaning | Typical Value |
|--------|---------|---------------|
| $t$ | A term in the query $q$ | — |
| $\text{tf}(t,d)$ | How many times term $t$ appears in document $d$ | 0, 1, 2, ... |
| $\text{IDF}(t)$ | $\log\frac{N - n(t) + 0.5}{n(t) + 0.5}$ — rarity of term $t$ across $N$ documents | Higher = rarer |
| $k_1$ | TF saturation parameter — controls how fast TF benefits plateau | 1.2–2.0 |
| $b$ | Length normalization — 0 = no normalization, 1 = full normalization | 0.75 |
| $|d|$ | Length of document $d$ (in tokens) | — |
| $\text{avgdl}$ | Average document length across the corpus | — |

### Intuition

- **IDF weight** ensures rare, discriminating terms (like "mitochondria") matter more than common ones (like "the").
- **The TF fraction** saturates: once a term appears enough, additional occurrences barely increase the score.
- **Length normalization** (controlled by $b$) penalizes long documents that match simply because they contain more words.

## BM25 Scoring in Action

Let's compute BM25 scores manually to see how the formula behaves. We'll vary term frequency (TF) and document length to observe:
- **TF saturation:** score flattens quickly after TF ≈ 5
- **Length normalization:** shorter documents get higher scores (same TF, fewer total words = stronger signal)

In [ ]:
import numpy as np

def bm25_score_manual(tf, df, N, dl, avgdl, k1=1.5, b=0.75):
    """Compute BM25 score for a single term in a single document."""
    idf = np.log((N - df + 0.5) / (df + 0.5) + 1)
    tf_component = (tf * (k1 + 1)) / (tf + k1 * (1 - b + b * dl / avgdl))
    return idf * tf_component

# Example: corpus of 10,000 docs, term appears in 50 docs
N, df, avgdl = 10000, 50, 200

print("BM25 Score by TF (term frequency) — showing saturation:")
print(f"{'TF':<6} {'Short doc (100)':<20} {'Avg doc (200)':<20} {'Long doc (500)'}")
print("─" * 60)
for tf in [0, 1, 2, 5, 10, 20, 50]:
    s1 = bm25_score_manual(tf, df, N, dl=100, avgdl=avgdl)
    s2 = bm25_score_manual(tf, df, N, dl=200, avgdl=avgdl)
    s3 = bm25_score_manual(tf, df, N, dl=500, avgdl=avgdl)
    print(f"{tf:<6} {s1:<20.4f} {s2:<20.4f} {s3:.4f}")

print("\n→ Notice: score saturates quickly (TF=5 vs TF=50 barely differs)")
print("→ Short documents get higher scores (length normalization at work)")

---
# §3 — Dense Retrieval Internals: Bi-Encoders and Semantic Space

## The Bi-Encoder Architecture

Dense retrieval uses a **bi-encoder**: two separate passes through a transformer encode the query and document into fixed-size vectors *independently*.

```
Query: "What causes climate change?"          Document: "Global warming is driven by..."
         ↓                                              ↓
   [Transformer Encoder]                         [Transformer Encoder]
         ↓                                              ↓
   q = [0.12, -0.34, 0.56, ...]                 d = [0.11, -0.31, 0.58, ...]
         ↓                                              ↓
                    similarity(q, d) = cos(q, d) = 0.97
```

### Why It Captures Semantics

During training, the model learns to map semantically similar texts to nearby points in vector space:
- "car" ≈ "automobile" ≈ "vehicle" → nearby vectors
- "bank" (financial) ⊥ "bank" (river) → distant vectors (in context)

### Why It Misses Keywords

Embeddings are **lossy compressions**. A 768-dimensional vector cannot perfectly encode every token in a 200-word document. Rare terms, codes, and proper nouns often get "averaged out":

- The embedding of *"Patient diagnosed with ICD-10 code J18.9"* may not differ meaningfully from *"Patient diagnosed with pneumonia"* — the model understands the concept but loses the exact code.
- Product IDs like "SKU-7829-BX" may be invisible in embedding space because the model never saw them in training.

### BGE-base-en-v1.5

Our embedding model (`BAAI/bge-base-en-v1.5`) is a 109M parameter BERT-based bi-encoder producing 768-dimensional embeddings. It was trained with:
- Contrastive learning on large-scale retrieval pairs
- Hard negative mining for better discrimination
- Instruction-tuned for retrieval tasks

## Embedding Similarity: Where Dense Shines and Fails

Let's encode query-document pairs with our BGE model and measure cosine similarity. We'll test:
- **Semantic matches** (different words, same meaning) — Dense should score high
- **Exact keyword matches** (codes, error IDs) — Dense may underperform

In [ ]:
from sentence_transformers.util import cos_sim

# Demonstrate where dense retrieval shines and where it fails
pairs = [
    # Semantic matches (dense wins)
    ("How does climate change affect weather?", "Global warming alters precipitation patterns"),
    ("car maintenance tips",                   "automobile repair and servicing guide"),
    ("treatment for high blood pressure",       "managing hypertension with medication"),
    # Exact keyword matches (dense may miss)
    ("COVID-19 transmission rate",              "COVID-19 has an R0 of approximately 2.5"),
    ("error code 0x80070005",                   "Fix for error code 0x80070005 in Windows"),
    ("ICD-10 J18.9",                            "ICD-10 code J18.9: Pneumonia, unspecified"),
]

print(f"{'Query':<40} {'Document':<50} {'Cosine Sim'}")
print("─" * 100)
for q, d in pairs:
    q_emb = embed_model.encode(q)
    d_emb = embed_model.encode(d)
    sim = cos_sim(q_emb, d_emb).item()
    print(f"{q:<40} {d:<50} {sim:.4f}")

print("\n→ Semantic pairs score high even without shared words.")
print("→ Exact keyword/code pairs may score lower than expected.")

---
# §4 — Why Fusion? Concrete Failure Cases

Here are real-world scenarios where each method **fails** and the other **succeeds**:

## Dense Retrieval Fails, BM25 Succeeds

| Query | Why Dense Fails | Why BM25 Succeeds |
|-------|-----------------|-------------------|
| "Error CVE-2024-3094" | Embedding dilutes the CVE ID | BM25 matches the exact string |
| "RFC 7519 specification" | RFC number gets lost in embedding | BM25 finds "RFC 7519" literally |
| "Python 3.12.4 changelog" | Version number is noise to embeddings | BM25 matches "3.12.4" exactly |

## BM25 Fails, Dense Retrieval Succeeds

| Query | Why BM25 Fails | Why Dense Succeeds |
|-------|----------------|--------------------|
| "How to fix a flat tire" | Doc says "repairing a punctured wheel" | Dense matches the semantics |
| "COVID-19 disease" | Doc says "coronavirus" throughout | Dense knows they're synonyms |
| "reduce server latency" | Doc discusses "improve response time" | Dense captures the equivalence |

## The Fusion Insight

**Neither method is strictly better.** Their failure modes are complementary:
- BM25 fails on **vocabulary mismatch** → Dense handles this.
- Dense fails on **exact term matching** → BM25 handles this.

By combining their ranked results, fusion retrieval inherits the strengths of both while mitigating the weaknesses of each.

---
# §5 — Reciprocal Rank Fusion (RRF)

## The Problem with Combining Scores Directly

BM25 scores range from 0 to ~30+. Cosine similarity ranges from -1 to 1. **You cannot simply add them** — the scales are incompatible.

Common solutions:
1. **Min-max normalization:** Rescale both to [0, 1]. But this is sensitive to outliers.
2. **Z-score normalization:** Subtract mean, divide by std. But assumes roughly normal distributions.
3. **Reciprocal Rank Fusion:** Ignore scores entirely — use only **ranks**.

## The RRF Formula

Given $n$ ranked lists (in our case, BM25 ranking and Dense ranking), the RRF score for document $d$ is:

$$\text{RRF}(d) = \sum_{i=1}^{n} \frac{1}{k + \text{rank}_i(d)}$$

where:
- $\text{rank}_i(d)$ is the rank of document $d$ in the $i$-th list (1-indexed)
- $k$ is a constant (typically **60**) that prevents the top-ranked document from dominating

## Why $k = 60$?

The original Cormack et al. (2009) paper tested values from 0 to 100. $k = 60$ was found to be robust across many benchmarks. The intuition:

- **$k = 0$:** Score = 1/rank. The #1 document gets 1.0, #2 gets 0.5. Too much weight on exact position.
- **$k = 60$:** Score = 1/(60 + rank). The #1 gets 1/61 ≈ 0.0164, #2 gets 1/62 ≈ 0.0161. Rankings are **smoothed** — a document ranked #1 vs #3 barely differs.
- **$k = 1000$:** Everything is nearly equal. Rankings barely matter.

## Why RRF is Robust

1. **Score-agnostic:** Works even when scores are on incompatible scales.
2. **Outlier-resistant:** A single very high BM25 score doesn't hijack results.
3. **Theoretically grounded:** Minimizes Kendall's tau distance to the "true" ranking under mild assumptions.
4. **No tuning needed:** $k = 60$ works well in practice across domains.

## Visualizing the RRF Constant $k$

The constant $k$ controls how much top-ranked documents dominate. Lower $k$ amplifies rank differences; higher $k$ flattens them. Let's compute RRF scores across different $k$ values to build intuition.

In [ ]:
import numpy as np

def rrf_score(rank, k=60):
    """Compute 1/(k + rank) for a single rank."""
    return 1.0 / (k + rank)

# Show how k affects score distribution
print("Effect of k on RRF scores (rank 1 through 10):")
print(f"{'Rank':<8}", end="")
for k_val in [0, 10, 60, 100]:
    print(f"{'k='+str(k_val):<14}", end="")
print()
print("─" * 64)

for rank in range(1, 11):
    print(f"{rank:<8}", end="")
    for k_val in [0, 10, 60, 100]:
        print(f"{rrf_score(rank, k_val):<14.6f}", end="")
    print()

print("\nRatio of rank-1 to rank-10 score:")
for k_val in [0, 10, 60, 100]:
    ratio = rrf_score(1, k_val) / rrf_score(10, k_val)
    print(f"  k={k_val:>3}: rank-1 is {ratio:.2f}x rank-10")

print("\n→ k=60 smooths rankings: rank-1 is only ~1.15x rank-10.")
print("→ k=0  is extreme: rank-1 is 10x rank-10.")

## RRF Worked Example

Let's walk through a concrete example. Two retrievers return different rankings for 5 documents. RRF combines them by summing reciprocal ranks. Documents ranked well by **both** retrievers rise to the top.

In [ ]:
# RRF worked example: merging BM25 and Dense rankings
print("=" * 70)
print("RRF Worked Example: Merging Two Ranked Lists")
print("=" * 70)

bm25_ranking  = ["Doc_A", "Doc_C", "Doc_B", "Doc_E", "Doc_D"]
dense_ranking = ["Doc_B", "Doc_A", "Doc_D", "Doc_C", "Doc_E"]
k = 60

print(f"\nBM25 ranking:  {bm25_ranking}")
print(f"Dense ranking: {dense_ranking}")
print(f"k = {k}")

# Compute RRF scores
all_docs = set(bm25_ranking) | set(dense_ranking)
rrf_scores = {}

print(f"\n{'Doc':<10} {'BM25 rank':<12} {'Dense rank':<12} {'BM25 RRF':<14} {'Dense RRF':<14} {'Total RRF'}")
print("─" * 72)

for doc in sorted(all_docs):
    bm25_rank = bm25_ranking.index(doc) + 1 if doc in bm25_ranking else len(bm25_ranking) + 1
    dense_rank = dense_ranking.index(doc) + 1 if doc in dense_ranking else len(dense_ranking) + 1
    bm25_rrf = 1.0 / (k + bm25_rank)
    dense_rrf = 1.0 / (k + dense_rank)
    total = bm25_rrf + dense_rrf
    rrf_scores[doc] = total
    print(f"{doc:<10} {bm25_rank:<12} {dense_rank:<12} {bm25_rrf:<14.6f} {dense_rrf:<14.6f} {total:.6f}")

# Sort by RRF score
fused_ranking = sorted(rrf_scores.items(), key=lambda x: x[1], reverse=True)
print(f"\nFused ranking: {[doc for doc, _ in fused_ranking]}")
print("\n→ Doc_A wins: ranked #1 by BM25 and #2 by Dense = consistently high.")
print("→ RRF rewards documents ranked well by MULTIPLE systems.")

---
# §6 — Implementation from Scratch

We'll build the complete fusion retrieval pipeline with:
1. A **synthetic knowledge base** designed to showcase BM25 vs Dense strengths
2. **BM25 index** using the `rank-bm25` library
3. **Dense index** using FAISS
4. **RRF fusion** function
5. Complete **hybrid retrieval** pipeline

## Building the Knowledge Base

Our synthetic corpus is carefully designed to expose the strengths of each retrieval method:

- **Docs 0–2:** Keyword-rich (CVE IDs, RFC numbers, PEP references) — BM25 territory
- **Docs 3–5:** Paraphrased/synonym-heavy content — Dense territory  
- **Docs 6–14:** Mixed content where both methods contribute

This design ensures our comparison experiments produce meaningful contrasts.

In [ ]:
# ── Synthetic Knowledge Base ──
# Designed to highlight BM25 vs Dense retrieval differences
# Each document has characteristics that favor one method over the other

KNOWLEDGE_BASE = [
    # --- Exact-keyword documents (BM25 shines) ---
    {
        "id": "doc_0",
        "text": "CVE-2024-3094 is a critical vulnerability in XZ Utils versions 5.6.0 and 5.6.1. "
                "The backdoor was inserted into the liblzma library through a sophisticated supply chain "
                "attack. The CVSS score is 10.0 (Critical). Systems running Fedora 40, Fedora Rawhide, "
                "openSUSE Tumbleweed, and Kali Linux may be affected. Remediation: downgrade to XZ Utils 5.4.x."
    },
    {
        "id": "doc_1",
        "text": "RFC 7519 defines the JSON Web Token (JWT) standard. A JWT consists of three parts: "
                "Header, Payload, and Signature, separated by dots. The header specifies the algorithm "
                "(e.g., HS256, RS256). Claims include 'iss' (issuer), 'exp' (expiration), 'sub' (subject). "
                "JWTs are Base64url-encoded and signed using HMAC or RSA."
    },
    {
        "id": "doc_2",
        "text": "Python 3.12 introduced several performance improvements. PEP 709 inlines comprehensions, "
                "reducing overhead by up to 2x. PEP 684 adds per-interpreter GIL for true parallelism. "
                "The 'type' statement (PEP 695) simplifies generic syntax. Error messages now suggest "
                "'import X' when NameError occurs. Startup time improved by 10-15%."
    },
    # --- Semantic documents (Dense retrieval shines) ---
    {
        "id": "doc_3",
        "text": "The novel coronavirus disease has fundamentally altered global public health strategies. "
                "The pathogen spreads primarily through respiratory droplets and aerosols. The basic "
                "reproduction number indicates each infected individual transmits to approximately 2-3 "
                "others. Vaccination campaigns have been the primary intervention, alongside social "
                "distancing measures and mask mandates."
    },
    {
        "id": "doc_4",
        "text": "Reducing server response latency requires a multi-pronged approach. Database query "
                "optimization through proper indexing and query planning can cut milliseconds from each "
                "request. Implementing a content delivery network distributes static assets closer to "
                "end users. Connection pooling prevents the overhead of establishing new database "
                "connections for each request. Caching frequently accessed data in memory with Redis "
                "or Memcached eliminates redundant computations."
    },
    {
        "id": "doc_5",
        "text": "Automobile maintenance extends vehicle longevity significantly. Regular oil changes every "
                "5,000 to 7,500 miles prevent engine wear. Tire rotation ensures even tread wear across "
                "all wheels. Brake pad inspection should occur at every servicing interval. The cooling "
                "system requires antifreeze flush every 30,000 miles. Transmission fluid replacement "
                "at manufacturer intervals prevents costly drivetrain failures."
    },
    # --- Documents where BOTH methods contribute ---
    {
        "id": "doc_6",
        "text": "FAISS (Facebook AI Similarity Search) is an open-source library for efficient similarity "
                "search of dense vectors. It supports multiple index types: IndexFlatL2 for exact search, "
                "IndexIVFFlat for approximate search with inverted file indexing, and IndexHNSW for "
                "hierarchical navigable small world graphs. GPU acceleration provides 5-10x speedup. "
                "For billion-scale datasets, product quantization (PQ) compresses vectors to 64 bytes."
    },
    {
        "id": "doc_7",
        "text": "The transformer architecture revolutionized natural language processing. Self-attention "
                "mechanisms allow each token to attend to every other token in the sequence, capturing "
                "long-range dependencies. Multi-head attention runs multiple attention functions in "
                "parallel, each learning different relationship patterns. Layer normalization and "
                "residual connections enable training of very deep models. The original paper "
                "'Attention Is All You Need' by Vaswani et al. (2017) introduced this architecture."
    },
    {
        "id": "doc_8",
        "text": "BM25 (Best Matching 25) is a probabilistic ranking function based on the binary "
                "independence model. It extends TF-IDF by adding term frequency saturation via parameter "
                "k1 (typically 1.2-2.0) and document length normalization via parameter b (typically 0.75). "
                "BM25 remains the default ranking function in Elasticsearch, Apache Solr, and most "
                "modern search engines due to its simplicity and effectiveness."
    },
    {
        "id": "doc_9",
        "text": "Retrieval-Augmented Generation (RAG) grounds large language model responses in external "
                "knowledge. The retrieval step fetches relevant documents from a corpus, which are then "
                "concatenated with the user query as context for the generator. This approach reduces "
                "hallucination, enables knowledge updates without retraining, and provides source "
                "attribution. Hybrid retrieval (combining sparse and dense methods) is the single most "
                "impactful improvement for RAG pipeline quality."
    },
    {
        "id": "doc_10",
        "text": "Climate change mitigation strategies include transitioning to renewable energy sources "
                "such as solar, wind, and hydroelectric power. Carbon capture and storage technology "
                "aims to remove CO2 directly from industrial emissions. Reforestation efforts can "
                "sequester approximately 3.6 billion tonnes of CO2 annually. International agreements "
                "like the Paris Accord set targets for limiting global temperature rise to 1.5°C above "
                "pre-industrial levels."
    },
    {
        "id": "doc_11",
        "text": "Kubernetes orchestrates containerized applications across clusters of machines. Pods are "
                "the smallest deployable units, containing one or more containers. Services provide stable "
                "network endpoints via ClusterIP, NodePort, or LoadBalancer types. Horizontal Pod "
                "Autoscaler adjusts replica count based on CPU utilization or custom metrics. "
                "ConfigMaps and Secrets manage application configuration and sensitive data respectively."
    },
    {
        "id": "doc_12",
        "text": "The human immune system consists of innate and adaptive components. Innate immunity "
                "provides immediate, non-specific defense through physical barriers, phagocytes, and "
                "inflammatory responses. Adaptive immunity develops targeted responses through T-cells "
                "and B-cells, producing antibodies specific to encountered pathogens. Immunological "
                "memory enables faster responses to previously encountered threats, which is the "
                "principle behind vaccination."
    },
    {
        "id": "doc_13",
        "text": "Punctured wheel repair on the roadside requires specific steps. First, engage the "
                "parking brake and place wheel chocks. Loosen lug nuts before jacking. Position the "
                "jack on the vehicle's designated lift point. Raise until the damaged wheel clears the "
                "ground. Remove lug nuts, swap the wheel for the spare, and hand-tighten in a star "
                "pattern. Lower the vehicle and torque lug nuts to specification."
    },
    {
        "id": "doc_14",
        "text": "Error code 0x80070005 in Windows indicates ACCESS_DENIED. Common causes include "
                "insufficient permissions for Windows Update, corrupted system files, or misconfigured "
                "security policies. Resolution steps: run 'sfc /scannow' to repair system files, "
                "reset Windows Update components using 'net stop wuauserv && net stop bits', "
                "and ensure the SYSTEM account has Full Control permissions on the SoftwareDistribution folder."
    },
]

texts = [doc["text"] for doc in KNOWLEDGE_BASE]
doc_ids = [doc["id"] for doc in KNOWLEDGE_BASE]

print(f"✓ Knowledge base: {len(KNOWLEDGE_BASE)} documents")
print(f"  Average length: {np.mean([len(t.split()) for t in texts]):.0f} words")
for doc in KNOWLEDGE_BASE:
    print(f"  {doc['id']}: {doc['text'][:70]}...")

## Step 1–2: Build BM25 Sparse Index and FAISS Dense Index

We build both indexes in sequence:
- **BM25:** Tokenize each document (lowercase + whitespace split), feed into `BM25Okapi` to build the inverted index and compute IDF weights.
- **FAISS:** Encode all documents with `bge-base-en-v1.5` (768-dim, L2-normalized), store in `IndexFlatIP` (inner product on normalized vectors = cosine similarity).

In [ ]:
from rank_bm25 import BM25Okapi

# ── Build BM25 Index ──
# Tokenize: lowercase + split on whitespace (simple but effective for English)
tokenized_corpus = [text.lower().split() for text in texts]

bm25_index = BM25Okapi(tokenized_corpus)

# Test it with a keyword-heavy query
test_query = "CVE-2024-3094 vulnerability"
tokenized_query = test_query.lower().split()
bm25_scores = bm25_index.get_scores(tokenized_query)

print(f"BM25 Index built: {len(tokenized_corpus)} documents")
print(f"\nTest query: '{test_query}'")
print(f"\n{'Doc ID':<10} {'BM25 Score':<14} {'First 60 chars'}")
print("─" * 80)
for idx in np.argsort(bm25_scores)[::-1]:
    if bm25_scores[idx] > 0:
        print(f"{doc_ids[idx]:<10} {bm25_scores[idx]:<14.4f} {texts[idx][:60]}...")

print("\n→ BM25 finds the CVE document instantly via exact string match.")

In [ ]:
import faiss

# ── Build Dense (FAISS) Index ──
# Encode all documents
doc_embeddings = embed_model.encode(texts, show_progress_bar=True, normalize_embeddings=True)
dim = doc_embeddings.shape[1]

# Create FAISS index (inner product on normalized vectors = cosine similarity)
faiss_index = faiss.IndexFlatIP(dim)
faiss_index.add(doc_embeddings.astype(np.float32))

print(f"FAISS Index built: {faiss_index.ntotal} vectors, dimension={dim}")

# Test with a semantic query
test_query_semantic = "How does the disease caused by the novel coronavirus spread?"
q_emb = embed_model.encode([test_query_semantic], normalize_embeddings=True).astype(np.float32)
scores, indices = faiss_index.search(q_emb, k=5)

print(f"\nTest query: '{test_query_semantic}'")
print(f"\n{'Rank':<6} {'Doc ID':<10} {'Cosine Sim':<14} {'First 60 chars'}")
print("─" * 86)
for rank, (idx, score) in enumerate(zip(indices[0], scores[0]), 1):
    print(f"{rank:<6} {doc_ids[idx]:<10} {score:<14.4f} {texts[idx][:60]}...")

print("\n→ Dense retrieval finds doc_3 ('novel coronavirus disease') semantically.")
print("  Note: the query uses no exact words from doc_3!")

## Step 3: Retrieval Functions

We wrap each retriever in a clean function that returns `(doc_index, score)` tuples sorted by relevance. This standardized interface makes fusion straightforward.

In [ ]:
def retrieve_bm25(query: str, top_k: int = 5) -> list:
    """Retrieve documents using BM25 sparse scoring.
    Returns list of (doc_index, score) sorted by score descending."""
    tokenized_query = query.lower().split()
    scores = bm25_index.get_scores(tokenized_query)
    ranked_indices = np.argsort(scores)[::-1][:top_k]
    return [(int(idx), float(scores[idx])) for idx in ranked_indices]

def retrieve_dense(query: str, top_k: int = 5) -> list:
    """Retrieve documents using dense embedding similarity (FAISS).
    Returns list of (doc_index, score) sorted by score descending."""
    q_emb = embed_model.encode([query], normalize_embeddings=True).astype(np.float32)
    scores, indices = faiss_index.search(q_emb, k=top_k)
    return [(int(idx), float(score)) for idx, score in zip(indices[0], scores[0])]

# Verify both work
q = "What is the BM25 ranking function?"
print(f"Query: '{q}'\n")

print("BM25 results:")
for idx, score in retrieve_bm25(q, top_k=3):
    print(f"  {doc_ids[idx]}: score={score:.4f}  |  {texts[idx][:60]}...")

print("\nDense results:")
for idx, score in retrieve_dense(q, top_k=3):
    print(f"  {doc_ids[idx]}: score={score:.4f}  |  {texts[idx][:60]}...")

print("\n✓ Both retrievers operational.")

## Core RRF Implementation

Now we implement the Reciprocal Rank Fusion algorithm. The function takes multiple ranked lists and merges them using the RRF formula:

$$\text{RRF}(d) = \sum_{i=1}^{n} \frac{1}{k + \text{rank}_i(d)}$$

Documents not present in a particular ranked list receive no contribution from that list (they are not penalized, just not boosted).

In [ ]:
from typing import List, Tuple, Dict

def reciprocal_rank_fusion(
    ranked_lists: List[List[Tuple[int, float]]],
    k: int = 60
) -> List[Tuple[int, float]]:
    """Reciprocal Rank Fusion over multiple ranked lists.

    Args:
        ranked_lists: List of ranked results, each a list of (doc_index, score)
                      in descending order of relevance.
        k: RRF constant (default 60).

    Returns:
        Fused ranking as list of (doc_index, rrf_score) in descending order.
    """
    rrf_scores: Dict[int, float] = {}

    for ranked_list in ranked_lists:
        for rank, (doc_idx, _original_score) in enumerate(ranked_list, start=1):
            if doc_idx not in rrf_scores:
                rrf_scores[doc_idx] = 0.0
            rrf_scores[doc_idx] += 1.0 / (k + rank)

    # Sort by RRF score descending
    fused = sorted(rrf_scores.items(), key=lambda x: x[1], reverse=True)
    return fused

# Test with a simple example
test_bm25 = [(0, 5.2), (2, 3.1), (1, 1.0)]
test_dense = [(1, 0.95), (0, 0.88), (3, 0.72)]

fused = reciprocal_rank_fusion([test_bm25, test_dense], k=60)
print("Test RRF fusion:")
print(f"  BM25 ranking:  {[(doc_ids[i], f'{s:.2f}') for i, s in test_bm25]}")
print(f"  Dense ranking: {[(doc_ids[i], f'{s:.2f}') for i, s in test_dense]}")
print(f"  Fused ranking: {[(doc_ids[i], f'{s:.6f}') for i, s in fused]}")
print("\n✓ RRF function working. Doc 0 tops because it's ranked highly in both lists.")

In [ ]:
def hybrid_retrieve(
    query: str,
    top_k: int = 5,
    retriever_top_k: int = 10,
    rrf_k: int = 60,
) -> List[Tuple[int, float]]:
    """Complete hybrid retrieval pipeline: BM25 + Dense + RRF.

    Args:
        query: User query string.
        top_k: Number of final results to return.
        retriever_top_k: Number of results from each individual retriever.
        rrf_k: RRF smoothing constant.

    Returns:
        Fused top-k results as list of (doc_index, rrf_score).
    """
    # Stage 1: Retrieve from both systems
    bm25_results = retrieve_bm25(query, top_k=retriever_top_k)
    dense_results = retrieve_dense(query, top_k=retriever_top_k)

    # Stage 2: Fuse with RRF
    fused_results = reciprocal_rank_fusion([bm25_results, dense_results], k=rrf_k)

    return fused_results[:top_k]

# Test the pipeline
test_query = "What is the JSON Web Token standard?"
results = hybrid_retrieve(test_query, top_k=5)

print(f"Query: '{test_query}'")
print(f"\n{'Rank':<6} {'Doc ID':<10} {'RRF Score':<14} {'Text Preview'}")
print("─" * 86)
for rank, (idx, score) in enumerate(results, 1):
    print(f"{rank:<6} {doc_ids[idx]:<10} {score:<14.6f} {texts[idx][:60]}...")

print("\n✓ Hybrid retrieval pipeline operational.")

---
# §7 — Head-to-Head Comparison: BM25 vs Dense vs Fusion

We now test all three methods on carefully chosen queries that reveal their strengths and weaknesses:

1. **Keyword-heavy queries** — where BM25 should dominate
2. **Semantic queries** — where Dense should dominate
3. **Mixed queries** — where Fusion should outperform both

In [ ]:
def compare_retrievers(query: str, top_k: int = 3):
    """Compare BM25, Dense, and Hybrid retrieval for a query."""
    bm25_results = retrieve_bm25(query, top_k=top_k)
    dense_results = retrieve_dense(query, top_k=top_k)
    hybrid_results = hybrid_retrieve(query, top_k=top_k)

    print(f"\n{'='*80}")
    print(f"Query: '{query}'")
    print(f"{'='*80}")

    for name, results in [("BM25 (Sparse)", bm25_results),
                          ("Dense (FAISS)", dense_results),
                          ("FUSION (RRF)",  hybrid_results)]:
        print(f"\n  {name}:")
        for rank, (idx, score) in enumerate(results, 1):
            marker = "" 
            print(f"    #{rank} {doc_ids[idx]} (score={score:.4f}): {texts[idx][:65]}...")

print("Comparison function ready.")

In [ ]:
print("\n" + "▓" * 80)
print("CATEGORY 1: KEYWORD-HEAVY QUERIES (BM25 advantage)")
print("▓" * 80)

keyword_queries = [
    "CVE-2024-3094 XZ Utils backdoor",
    "error code 0x80070005 Windows",
    "RFC 7519 JWT specification",
]

for q in keyword_queries:
    compare_retrievers(q)

print("\n→ BM25 dominates on exact keyword/code queries.")
print("→ Dense retrieval may rank the correct doc lower.")
print("→ Fusion inherits BM25's strength here — correct doc still tops.")

In [ ]:
print("\n" + "▓" * 80)
print("CATEGORY 2: SEMANTIC QUERIES (Dense advantage)")
print("▓" * 80)

semantic_queries = [
    "How does the respiratory illness caused by the new coronavirus spread between people?",
    "How to fix a flat tire on the roadside",
    "Ways to make web applications respond faster",
]

for q in semantic_queries:
    compare_retrievers(q)

print("\n→ Dense retrieval shines: it finds semantically relevant docs despite no keyword overlap.")
print("→ BM25 struggles with paraphrases and synonym-heavy queries.")
print("→ Fusion inherits Dense's strength — semantic matches still surface.")

In [ ]:
print("\n" + "▓" * 80)
print("CATEGORY 3: MIXED QUERIES (Fusion advantage)")
print("▓" * 80)

mixed_queries = [
    "BM25 ranking algorithm for search engines",
    "RAG retrieval augmented generation hybrid search",
    "FAISS vector similarity search for embeddings",
]

for q in mixed_queries:
    compare_retrievers(q)

print("\n→ Mixed queries benefit from both exact keyword AND semantic matching.")
print("→ Fusion surfaces the best results from both worlds.")

---
# §8 — Weighted Fusion: Tuning the Sparse/Dense Balance

RRF treats all ranked lists equally. But sometimes you want to **weight** one retriever more than the other:

- **Technical documentation** with many codes/IDs → weight BM25 higher
- **Conversational FAQ** with paraphrases → weight Dense higher
- **General corpus** → equal weights (standard RRF)

## Weighted RRF

We extend RRF with per-retriever weights:

$$\text{wRRF}(d) = \sum_{i=1}^{n} w_i \cdot \frac{1}{k + \text{rank}_i(d)}$$

where $w_i$ is the weight for the $i$-th retriever.

In [ ]:
def weighted_rrf(
    ranked_lists: List[List[Tuple[int, float]]],
    weights: List[float],
    k: int = 60,
) -> List[Tuple[int, float]]:
    """Weighted Reciprocal Rank Fusion.

    Args:
        ranked_lists: List of ranked results from each retriever.
        weights: Per-retriever weights (e.g., [0.4, 0.6] for BM25=0.4, Dense=0.6).
        k: RRF constant.

    Returns:
        Weighted fused ranking.
    """
    assert len(ranked_lists) == len(weights), "Must have one weight per ranked list"

    rrf_scores: Dict[int, float] = {}

    for weight, ranked_list in zip(weights, ranked_lists):
        for rank, (doc_idx, _) in enumerate(ranked_list, start=1):
            if doc_idx not in rrf_scores:
                rrf_scores[doc_idx] = 0.0
            rrf_scores[doc_idx] += weight * (1.0 / (k + rank))

    return sorted(rrf_scores.items(), key=lambda x: x[1], reverse=True)


def hybrid_retrieve_weighted(
    query: str,
    top_k: int = 5,
    bm25_weight: float = 0.5,
    dense_weight: float = 0.5,
    retriever_top_k: int = 10,
    rrf_k: int = 60,
) -> List[Tuple[int, float]]:
    """Weighted hybrid retrieval pipeline."""
    bm25_results = retrieve_bm25(query, top_k=retriever_top_k)
    dense_results = retrieve_dense(query, top_k=retriever_top_k)
    fused = weighted_rrf([bm25_results, dense_results],
                         weights=[bm25_weight, dense_weight], k=rrf_k)
    return fused[:top_k]

print("✓ Weighted RRF functions defined.")

In [ ]:
# Show how different weights affect results
query = "CVE-2024-3094 supply chain security vulnerability"
weight_configs = [
    (0.8, 0.2, "BM25-heavy (0.8/0.2)"),
    (0.5, 0.5, "Equal      (0.5/0.5)"),
    (0.2, 0.8, "Dense-heavy (0.2/0.8)"),
]

print(f"Query: '{query}'\n")

for bm25_w, dense_w, label in weight_configs:
    results = hybrid_retrieve_weighted(query, top_k=3,
                                       bm25_weight=bm25_w, dense_weight=dense_w)
    print(f"{label}:")
    for rank, (idx, score) in enumerate(results, 1):
        print(f"  #{rank} {doc_ids[idx]} (wRRF={score:.6f}): {texts[idx][:55]}...")
    print()

print("→ BM25-heavy weight favors the exact CVE match.")
print("→ Dense-heavy weight might surface related security docs.")
print("→ Tuning weights lets you control the keyword vs semantic balance.")

In [ ]:
# Weight sensitivity: how does top-1 doc change across the weight spectrum?
queries_for_analysis = [
    "RFC 7519 JWT claims",
    "How does the immune system fight disease?",
    "transformer architecture attention mechanism",
]

weight_range = np.arange(0, 1.05, 0.1)

print("Weight sensitivity analysis: Top-1 document at each BM25 weight")
print("(BM25 weight from 0.0=Dense-only to 1.0=BM25-only)\n")

for q in queries_for_analysis:
    print(f"Query: '{q}'")
    top1_docs = []
    for bw in weight_range:
        res = hybrid_retrieve_weighted(q, top_k=1,
                                       bm25_weight=bw, dense_weight=1-bw)
        top1_docs.append(doc_ids[res[0][0]])

    for bw, d in zip(weight_range, top1_docs):
        print(f"  BM25_w={bw:.1f}: {d}", end="")
        if bw in [0.0, 0.5, 1.0]:
            print("  ←", end="")
        print()
    print()

print("→ The optimal weight depends on query type.")
print("→ In practice, 0.5/0.5 (equal) is a strong default.")

---
# §9 — Complete RAG Pipeline with Fusion Retrieval

Now we wire everything together: **query → hybrid retrieval → context assembly → LLM generation**.

This is the full Retrieval-Augmented Generation pipeline with fusion retrieval as the retrieval backbone.

In [ ]:
def fusion_rag(
    query: str,
    top_k: int = 3,
    bm25_weight: float = 0.5,
    dense_weight: float = 0.5,
    max_new_tokens: int = 512,
) -> str:
    """Full RAG pipeline with fusion retrieval.

    1. Retrieve top-k documents using weighted hybrid retrieval
    2. Assemble context from retrieved documents
    3. Generate answer with the LLM grounded in context
    """
    # Stage 1: Hybrid Retrieval
    results = hybrid_retrieve_weighted(
        query, top_k=top_k,
        bm25_weight=bm25_weight, dense_weight=dense_weight
    )

    # Stage 2: Context Assembly
    context_parts = []
    for i, (doc_idx, rrf_score) in enumerate(results, 1):
        context_parts.append(f"[Document {i} | {doc_ids[doc_idx]} | RRF={rrf_score:.6f}]\n{texts[doc_idx]}")
    context = "\n\n".join(context_parts)

    # Stage 3: Prompt Construction & Generation
    prompt = (
        f"You are a helpful assistant. Answer the question based ONLY on the provided context. "
        f"If the context doesn't contain enough information, say so.\n\n"
        f"Context:\n{context}\n\n"
        f"Question: {query}\n\n"
        f"Answer:"
    )

    answer = generate(prompt, max_new_tokens=max_new_tokens)

    return answer, results, context

print("✓ Fusion RAG pipeline defined.")

In [ ]:
# RAG Example 1: Keyword-heavy query
query = "What is CVE-2024-3094 and how do I remediate it?"
answer, results, context = fusion_rag(query, top_k=3)

print(f"Query: {query}")
print(f"\nRetrieved documents:")
for rank, (idx, score) in enumerate(results, 1):
    print(f"  #{rank} {doc_ids[idx]} (RRF={score:.6f})")
print(f"\nGenerated Answer:\n{answer}")

In [ ]:
# RAG Example 2: Semantic query (no exact keyword matches with target doc)
query = "How does the respiratory illness from the new pathogen transmit between people?"
answer, results, context = fusion_rag(query, top_k=3)

print(f"Query: {query}")
print(f"\nRetrieved documents:")
for rank, (idx, score) in enumerate(results, 1):
    print(f"  #{rank} {doc_ids[idx]} (RRF={score:.6f})")
print(f"\nGenerated Answer:\n{answer}")

In [ ]:
# RAG Example 3: Mixed query — benefits from both retrieval types
query = "How does FAISS compare to other approaches for nearest neighbor search in RAG pipelines?"
answer, results, context = fusion_rag(query, top_k=3)

print(f"Query: {query}")
print(f"\nRetrieved documents:")
for rank, (idx, score) in enumerate(results, 1):
    print(f"  #{rank} {doc_ids[idx]} (RRF={score:.6f})")
print(f"\nGenerated Answer:\n{answer}")

## Alternative: Score-Based Fusion (Min-Max Normalization)

While RRF is rank-based and ignores raw scores, sometimes you want to preserve score magnitude. For comparison, here's the score-normalization approach used in the original notebook:

1. Normalize BM25 scores to [0, 1] using min-max
2. Normalize Dense scores to [0, 1] (note: cosine distances need inversion)
3. Weighted combination: `alpha × dense_score + (1-alpha) × bm25_score`

**Trade-offs:**
- Score-based fusion preserves magnitude information but is sensitive to score distributions
- RRF is more robust and doesn't require calibrated scores
- In practice, RRF performs as well or better than score-based fusion in most benchmarks

In [ ]:
def score_based_fusion(
    query: str, top_k: int = 5, alpha: float = 0.5, retriever_top_k: int = 10
) -> List[Tuple[int, float]]:
    """Score-based fusion with min-max normalization.
    alpha: weight for dense scores (1-alpha for BM25)."""
    bm25_results = retrieve_bm25(query, top_k=retriever_top_k)
    dense_results = retrieve_dense(query, top_k=retriever_top_k)

    # Collect all unique docs
    all_docs = set()
    bm25_map = {}
    dense_map = {}
    for idx, score in bm25_results:
        all_docs.add(idx)
        bm25_map[idx] = score
    for idx, score in dense_results:
        all_docs.add(idx)
        dense_map[idx] = score

    # Min-max normalize
    def normalize(scores_dict):
        if not scores_dict:
            return {}
        vals = list(scores_dict.values())
        mn, mx = min(vals), max(vals)
        rng = mx - mn if mx - mn > 1e-8 else 1e-8
        return {k: (v - mn) / rng for k, v in scores_dict.items()}

    bm25_norm = normalize(bm25_map)
    dense_norm = normalize(dense_map)

    # Combine
    combined = {}
    for idx in all_docs:
        b = bm25_norm.get(idx, 0.0)
        d = dense_norm.get(idx, 0.0)
        combined[idx] = alpha * d + (1 - alpha) * b

    return sorted(combined.items(), key=lambda x: x[1], reverse=True)[:top_k]


# Compare RRF vs Score-based fusion
query = "transformer architecture attention mechanism paper"
rrf_results = hybrid_retrieve(query, top_k=3)
score_results = score_based_fusion(query, top_k=3, alpha=0.5)

print(f"Query: '{query}'\n")
print("RRF Fusion:")
for rank, (idx, score) in enumerate(rrf_results, 1):
    print(f"  #{rank} {doc_ids[idx]} ({score:.6f}): {texts[idx][:55]}...")
print("\nScore-Based Fusion (alpha=0.5):")
for rank, (idx, score) in enumerate(score_results, 1):
    print(f"  #{rank} {doc_ids[idx]} ({score:.4f}): {texts[idx][:55]}...")

print("\n→ Both methods find the right docs; RRF is more robust across query types.")

## Retrieval Quality Analysis

Let's run a systematic evaluation across all query types to see which method "wins" most often.

In [ ]:
# Systematic evaluation across query types
eval_queries = [
    # (query, expected_top_doc, category)
    ("CVE-2024-3094 XZ Utils",                                      "doc_0",  "keyword"),
    ("error code 0x80070005",                                        "doc_14", "keyword"),
    ("RFC 7519 JWT",                                                 "doc_1",  "keyword"),
    ("Python 3.12 performance PEP 709",                              "doc_2",  "keyword"),
    ("How does the novel coronavirus spread?",                       "doc_3",  "semantic"),
    ("repairing a punctured wheel on the road",                      "doc_13", "semantic"),
    ("ways to speed up web server responses",                        "doc_4",  "semantic"),
    ("car servicing and vehicle longevity",                          "doc_5",  "semantic"),
    ("BM25 probabilistic ranking function in search",                "doc_8",  "mixed"),
    ("RAG hybrid retrieval pipeline improvement",                    "doc_9",  "mixed"),
    ("FAISS approximate nearest neighbor vector search",             "doc_6",  "mixed"),
    ("Kubernetes container orchestration pod autoscaling",           "doc_11", "mixed"),
]

results_table = {"BM25": 0, "Dense": 0, "Fusion": 0}
category_results = {"keyword": {"BM25": 0, "Dense": 0, "Fusion": 0},
                    "semantic": {"BM25": 0, "Dense": 0, "Fusion": 0},
                    "mixed": {"BM25": 0, "Dense": 0, "Fusion": 0}}

print(f"{'Query':<55} {'Expected':<10} {'BM25#1':<10} {'Dense#1':<10} {'Fusion#1'}")
print("─" * 100)

for query, expected, category in eval_queries:
    bm25_top = doc_ids[retrieve_bm25(query, top_k=1)[0][0]]
    dense_top = doc_ids[retrieve_dense(query, top_k=1)[0][0]]
    fusion_top = doc_ids[hybrid_retrieve(query, top_k=1)[0][0]]

    bm25_hit = "✓" if bm25_top == expected else "✗"
    dense_hit = "✓" if dense_top == expected else "✗"
    fusion_hit = "✓" if fusion_top == expected else "✗"

    if bm25_top == expected: results_table["BM25"] += 1; category_results[category]["BM25"] += 1
    if dense_top == expected: results_table["Dense"] += 1; category_results[category]["Dense"] += 1
    if fusion_top == expected: results_table["Fusion"] += 1; category_results[category]["Fusion"] += 1

    q_short = query[:53] + ".." if len(query) > 55 else query
    print(f"{q_short:<55} {expected:<10} {bm25_hit+' '+bm25_top:<10} {dense_hit+' '+dense_top:<10} {fusion_hit+' '+fusion_top}")

print(f"\nOverall hits (top-1 = expected):")
for method, hits in results_table.items():
    print(f"  {method}: {hits}/{len(eval_queries)} ({100*hits/len(eval_queries):.0f}%)")

print(f"\nBy category:")
for cat, methods in category_results.items():
    n = sum(1 for q, e, c in eval_queries if c == cat)
    print(f"  {cat}: BM25={methods['BM25']}/{n}, Dense={methods['Dense']}/{n}, Fusion={methods['Fusion']}/{n}")

print("\n→ Fusion is the most consistently correct across ALL query categories.")

## 🏋️ Exercises

Put your understanding to the test:

**Exercise 1 — Explore:** swap out the retriever/chunker and measure the impact on recall. Document what changes and why.

**Exercise 2 — Build:** add a new document type and test retrieval quality. Compare your implementation to the one in this notebook.

**Exercise 3 — Extend:** implement a simple version of the algorithm from scratch.

---
# §10 — Synthesis: Fusion as the Highest-Impact RAG Improvement

## Why Fusion Retrieval Matters

Research consistently shows that **hybrid/fusion retrieval is the single most impactful improvement** you can make to a RAG pipeline:

- **Microsoft Research (2023):** Hybrid search improved recall@10 by 10-20% vs. dense-only on production Bing queries.
- **Anthropic (2024):** Recommended hybrid retrieval as the #1 technique for RAG quality.
- **LlamaIndex benchmarks:** Hybrid consistently outperforms single-method retrieval across 8 benchmark datasets.

## When to Use Each Approach

| Scenario | Recommendation |
|----------|---------------|
| General-purpose RAG | **Fusion (RRF)** with equal weights |
| Code search / technical docs | Fusion with **BM25-heavy** weights (0.6-0.8) |
| Conversational / FAQ | Fusion with **Dense-heavy** weights (0.6-0.8) |
| Exact ID/code lookup only | BM25-only may suffice |
| Multilingual / cross-lingual | Dense-only (BM25 is language-specific) |

## Production Patterns

### 1. Parallel Retrieval
Run BM25 and Dense retrieval in parallel (they're independent). RRF adds negligible overhead.

### 2. Retriever Cascade
Use BM25 for fast first-pass (100 candidates), then rerank with a cross-encoder, then fuse with dense results.

### 3. Query-Adaptive Weights
Classify the query type (keyword vs. semantic) and adjust weights dynamically. A simple heuristic: if the query contains proper nouns, codes, or IDs → increase BM25 weight.

### 4. RRF with Multiple Retrievers
RRF scales to $n$ ranked lists. You can fuse BM25 + Dense + Reranker + Knowledge Graph results seamlessly.

## Key Takeaways

1. **BM25 and Dense retrieval have complementary blind spots** — keywords vs. semantics.
2. **RRF is the cleanest fusion method** — rank-based, score-agnostic, outlier-resistant, no tuning needed.
3. **k=60 works well** out of the box; equal weights are a strong default.
4. **Fusion retrieval should be your baseline** for any RAG system. Start here, then add reranking, query expansion, etc.
5. **The implementation is simple** — two retrievers, one formula, massive quality gains.

In [ ]:
# Final summary: the complete fusion retrieval toolkit
print("=" * 70)
print("FUSION RETRIEVAL TOOLKIT — SUMMARY")
print("=" * 70)
print()
print("Components built in this notebook:")
print("  1. BM25 sparse index (rank-bm25)")
print("  2. Dense vector index (FAISS + bge-base-en-v1.5)")
print("  3. Reciprocal Rank Fusion (RRF) — score = Σ 1/(k + rank)")
print("  4. Weighted RRF — configurable sparse/dense balance")
print("  5. Score-based fusion alternative (min-max normalization)")
print("  6. Complete RAG pipeline with fusion retrieval")
print()
print("Key findings:")
print("  • BM25 dominates on keyword/code queries")
print("  • Dense dominates on semantic/paraphrase queries")
print("  • Fusion is most CONSISTENT across all query types")
print("  • RRF(k=60) works well with no tuning")
print("  • Equal weights (0.5/0.5) are a strong default")
print()
print("Production recommendations:")
print("  • Always start with fusion retrieval as your RAG baseline")
print("  • Run BM25 and Dense in parallel for zero latency overhead")
print("  • Tune weights per domain if needed (codes→BM25, chat→Dense)")
print("  • Layer reranking on top for additional quality gains")

---

## 🧭 Navigation

⬅️ **Previous:** [Explainable Retrieval](https://colab.research.google.com/github/cyruslayo/castalia/blob/main/rag/explainable_retrieval.ipynb) | ➡️ **Next:** [Graph Rag](https://colab.research.google.com/github/cyruslayo/castalia/blob/main/rag/graph_rag.ipynb)